# Lab 11: Grid Localization using Bayes Filter (Virtual Robot)

### <span style="color:rgb(0,150,0)">It is recommended that you close any heavy-duty applications running on your system while working on this lab.</span>

<hr>


In [1]:
%load_ext autoreload
%autoreload 2

import traceback
from notebook_utils import *
from Traj import *
import asyncio
from localization_extras import Localization

# Setup Logger
LOG = get_logger('demo_notebook.log')

# Init GUI and Commander
gui = GET_GUI()
cmdr = gui.launcher.commander

gui.show()

# Start the simulator
START_SIM()

# Start the plotter
START_PLOTTER()

2026-04-25 23:50:44,752 | INFO     |: Logger demo_notebook.log initialized.


/Users/y1hhnn/.pyenv/versions/FastRobots_ble/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


TwoByTwoLayout(children=(Label(value='Simulator', layout=Layout(grid_area='top-left', width='80px')), HBox(chi…

Loading Flatland...
Initializing pygame framework...


/Users/y1hhnn/.pyenv/versions/FastRobots_ble/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
# Initialize Robot to communicate with the virtual robot and plotter
robot = VirtualRobot(cmdr)

# Initialize mapper
# Requires a VirtualRobot object as a parameter
mapper = Mapper(robot)

# Initialize your BaseLocalization object
# Requires a VirtualRobot object and a Mapper object as parameters
loc = Localization(robot, mapper)

## Plot Map
cmdr.plot_map()

2026-04-25 23:50:57,409 | INFO     |:  | Number of observations per grid cell: 18
2026-04-25 23:50:57,409 | INFO     |:  | Precaching Views...


/Users/y1hhnn/Desktop/ECE4160/FastRobots-sim-release/localization.py:150: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(distance_intersections_tt), intersections_tt[np.nanargmin(distance_intersections_tt)]


2026-04-25 23:50:58,369 | INFO     |:  | Precaching Time: 0.960 secs
2026-04-25 23:50:58,370 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-25 23:50:58,370 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107


# Run the Bayes Filter
The cells below utilizes the member functions of class **Localization** (defined in [localization_extras.py](../localization_extras.py)) in each iteration of the Bayes filter algorithm to localize the robot in the grid map. <br>

In [3]:
# Reset Robot and Plots
robot.reset()
cmdr.reset_plotter()

# Init Uniform Belief
loc.init_grid_beliefs()

# Get Observation Data by executing a 360 degree rotation motion
loc.get_observation_data()

# Run Update Step
loc.update_step()
loc.print_update_stats(plot_data=True)

# Plot Odom and GT
current_odom, current_gt = robot.get_pose()
cmdr.plot_gt(current_gt[0], current_gt[1])
cmdr.plot_odom(current_odom[0], current_odom[1])

2026-04-25 23:51:04,600 | INFO     |: Initializing beliefs with a Uniform Distribution
2026-04-25 23:51:04,601 | INFO     |: Uniform Belief with each cell value: 0.00051440329218107
2026-04-25 23:51:07,572 | INFO     |: Update Step
2026-04-25 23:51:07,573 | INFO     |:      | Update Time: 0.001 secs
2026-04-25 23:51:07,573 | INFO     |: ---------- UPDATE STATS -----------
2026-04-25 23:51:07,588 | INFO     |: GT index      : (6, 4, 9)
2026-04-25 23:51:07,589 | INFO     |: Bel index     : (np.int64(5), np.int64(4), np.int64(9)) with prob = 0.8828250
2026-04-25 23:51:07,589 | INFO     |: Bel_bar prob at index = 0.00051440329218107
2026-04-25 23:51:07,590 | INFO     |: GT            : (0.000, 0.000, 360.000)
2026-04-25 23:51:07,590 | INFO     |: Belief        : (0.000, 0.000, 10.000)
2026-04-25 23:51:07,590 | INFO     |: POS ERROR     : (-0.000, -0.000, 350.000)
2026-04-25 23:51:07,591 | INFO     |: ---------- UPDATE STATS -----------


In [4]:
# Initialize the Trajectory object
traj = Trajectory(loc)

# Run through each motion steps
for t in range(0, traj.total_time_steps):
    print("\n\n-----------------", t, "-----------------")
    
    prev_odom, current_odom, prev_gt, current_gt = traj.execute_time_step(t)
        
    # Prediction Step
    loc.prediction_step(current_odom, prev_odom)
    loc.print_prediction_stats(plot_data=True)
    
    # Get Observation Data by executing a 360 degree rotation motion
    loc.get_observation_data()
    
    # Update Step
    loc.update_step()
    loc.print_update_stats(plot_data=True)

# Uncomment the below line to wait for keyboard input between each iteration.
#   input("Press Enter to Continue")
        
    print("-------------------------------------")



----------------- 0 -----------------
2026-04-25 23:51:14,196 | INFO     |: Prediction Step
2026-04-25 23:51:14,216 | INFO     |:  | Prediction Time: 0.020 secs
2026-04-25 23:51:14,217 | INFO     |: ---------- PREDICTION STATS -----------
2026-04-25 23:51:14,228 | INFO     |: GT index         : (6, 3, 7)
2026-04-25 23:51:14,228 | INFO     |: Prior Bel index  : (np.int64(6), np.int64(6), np.int64(7)) with prob = 0.0905117
2026-04-25 23:51:14,229 | INFO     |: POS ERROR        : (-0.018, -0.697, -9.057)
2026-04-25 23:51:14,229 | INFO     |: ---------- PREDICTION STATS -----------
2026-04-25 23:51:17,203 | INFO     |: Update Step
2026-04-25 23:51:17,204 | INFO     |:      | Update Time: 0.001 secs
2026-04-25 23:51:17,205 | INFO     |: ---------- UPDATE STATS -----------
2026-04-25 23:51:17,211 | INFO     |: GT index      : (6, 3, 7)
2026-04-25 23:51:17,212 | INFO     |: Bel index     : (np.int64(6), np.int64(4), np.int64(6)) with prob = 1.0
2026-04-25 23:51:17,212 | INFO     |: Bel_bar 

Traceback (most recent call last):
  File "/Users/y1hhnn/Desktop/ECE4160/FastRobots-sim-release/src/plotter.py", line 263, in keyPressEvent
    if event.key() == Qt.Key_Escape:
                      ^^^^^^^^^^^^^
AttributeError: type object 'Qt' has no attribute 'Key_Escape'
Traceback (most recent call last):
  File "/Users/y1hhnn/Desktop/ECE4160/FastRobots-sim-release/src/plotter.py", line 263, in keyPressEvent
    if event.key() == Qt.Key_Escape:
                      ^^^^^^^^^^^^^
AttributeError: type object 'Qt' has no attribute 'Key_Escape'
Traceback (most recent call last):
  File "/Users/y1hhnn/Desktop/ECE4160/FastRobots-sim-release/src/plotter.py", line 263, in keyPressEvent
    if event.key() == Qt.Key_Escape:
                      ^^^^^^^^^^^^^
AttributeError: type object 'Qt' has no attribute 'Key_Escape'
Traceback (most recent call last):
  File "/Users/y1hhnn/Desktop/ECE4160/FastRobots-sim-release/src/plotter.py", line 263, in keyPressEvent
    if event.key() == Qt.Key_Esc

In [5]:
# Start the simulator
STOP_SIM()

# Start the plotter
STOP_PLOTTER()

2026-04-25 23:54:27,831 | INFO     |: Simulator is stopped
2026-04-25 23:54:27,833 | INFO     |: Plotter is stopped
